In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 07 — Per-tile building attributes (enrichment)

Enriches existing per-tile validation outputs with reference building
density and average size, computed via centroid-based assignment.

**Non-destructive:** the original `vector_metrics_tiles_all_datasets.parquet`
is never modified. Results are written to:
- `outputs/metrics/<city>/vector_metrics_tiles_enriched.parquet` — per city
- `outputs/scratch/per_tile_enriched_all_cities.csv` — combined flat CSV

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

DATA_DIR     = PROJECT_ROOT / cfg['data_dir']
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
OUT_SCRATCH  = PROJECT_ROOT / 'outputs' / 'scratch'
OUT_SCRATCH.mkdir(parents=True, exist_ok=True)

SENTINEL = 'vector_metrics_tiles_all_datasets.parquet'
ENRICHED = 'vector_metrics_tiles_enriched.parquet'

tracker_path = PROJECT_ROOT / cfg['aoi_tracker']
tracker = pd.read_csv(tracker_path, dtype=str)
tracker.columns = tracker.columns.str.strip()
tracker = tracker.apply(lambda c: c.str.strip() if c.dtype == object else c)

_suit = next((c for c in tracker.columns if 'suitable' in c.lower()), None)
if _suit:
    tracker = tracker[tracker[_suit].str.lower() == 'yes']

_folder_col = 'dataset_folder_name'
_ref_col    = next((c for c in tracker.columns if 'reference' in c.lower() and 'file' in c.lower()), None)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'METRICS_ROOT : {METRICS_ROOT}  exists={METRICS_ROOT.exists()}')
print(f'Tracker columns : {tracker.columns.tolist()}')
print(f'Ref file col    : {_ref_col!r}  (None = will use glob fallback for all cities)')

In [ ]:
def load_tiles(city_slug: str, data_dir: Path):
    """Load tile grid GPKG written by BaseValidationRunner._save_tiles()."""
    tiles_path = data_dir / city_slug / 'tiles' / f'{city_slug.lower()}_tiles.gpkg'
    if not tiles_path.exists():
        return None
    return gpd.read_file(tiles_path)


def load_reference(city_slug: str, data_dir: Path, ref_spec) -> gpd.GeoDataFrame | None:
    """Load reference building footprints for a city.

    Tries tracker-specified filenames first; falls back to globbing the
    city's vector/ directory for common reference file patterns when
    ref_spec is None or the specified files aren't found.
    """
    vec_dir = data_dir / city_slug / 'vector'
    if not vec_dir.exists():
        return None

    candidates = []

    # Primary: tracker-specified filename(s)
    if ref_spec and str(ref_spec).strip() not in ('', 'nan', 'None'):
        for name in str(ref_spec).split('|'):
            name = name.strip()
            if name:
                candidates.append(vec_dir / name)

    # Fallback: glob for common reference file patterns
    if not candidates:
        for pat in ('*_ref.*', '*_reference.*', '*hotosm*', '*worldbank*', '*sn7*'):
            candidates.extend(vec_dir.glob(pat))
        # Exclude candidate-dataset files
        candidates = [
            p for p in candidates
            if not any(k in p.name.lower()
                       for k in ('overture', 'gba', 'globfp', 'obt', 'tempo', 'wsf'))
        ]

    gdfs = []
    for p in candidates:
        if not p.exists():
            continue
        try:
            gdfs.append(gpd.read_file(p) if p.suffix.lower() != '.parquet'
                        else gpd.read_parquet(p))
        except Exception as e:
            print(f'  [warn] could not read {p.name}: {e}')

    if not gdfs:
        return None
    if len(gdfs) == 1:
        return gdfs[0]
    combined = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    if combined.crs is None:
        combined = combined.set_crs('EPSG:4326')
    return combined


def enrich_tile_metrics(tile_metrics_df: pd.DataFrame,
                        tiles_gdf: gpd.GeoDataFrame,
                        ref_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Add 4 centroid-based building attribute columns to tile_metrics_df.

    Mirrors the logic in src/metrics/vector/tile_metrics.py but operates
    on all tiles at once via a single spatial join for efficiency.
    """
    if ref_gdf.crs != tiles_gdf.crs:
        ref_gdf = ref_gdf.to_crs(tiles_gdf.crs)

    # Centroid layer with area pre-computed in projected CRS
    ref_pts = ref_gdf.copy()
    ref_pts['area_m2'] = ref_gdf.geometry.area
    ref_pts['geometry'] = ref_gdf.geometry.centroid

    # Spatial join: centroid within tile
    joined = gpd.sjoin(
        ref_pts[['geometry', 'area_m2']],
        tiles_gdf[['tile_id', 'geometry']],
        how='inner',
        predicate='within',
    )

    per_tile = joined.groupby('tile_id').agg(
        ref_building_count_centroid=('area_m2', 'count'),
        mean_ref_building_area_m2=('area_m2', 'mean'),
    ).reset_index()

    tiles_area = tiles_gdf[['tile_id', 'geometry']].copy()
    tiles_area['tile_area_km2'] = tiles_area.geometry.area / 1e6

    merged = tiles_area[['tile_id', 'tile_area_km2']].merge(per_tile, on='tile_id', how='left')
    merged['ref_building_count_centroid'] = (
        merged['ref_building_count_centroid'].fillna(0).astype(int)
    )
    merged['ref_building_density_per_km2'] = (
        merged['ref_building_count_centroid']
        / merged['tile_area_km2'].replace(0, np.nan)
    )

    return tile_metrics_df.merge(
        merged.drop(columns='geometry', errors='ignore'),
        on='tile_id',
        how='left',
    )

In [ ]:
NEW_COLS = ['tile_area_km2', 'ref_building_count_centroid',
            'mean_ref_building_area_m2', 'ref_building_density_per_km2']

# Find all processed cities (have the sentinel parquet)
city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(SENTINEL))
print(f'Found {len(city_dirs)} processed cities\n')

all_enriched = []
skipped = []

for city_dir in city_dirs:
    city = city_dir.name
    sentinel_path = city_dir / SENTINEL
    enriched_path = city_dir / ENRICHED

    # Load existing tile metrics
    try:
        tile_df = pd.read_parquet(sentinel_path)
    except Exception as e:
        print(f'[SKIP] {city}: cannot read parquet — {e}')
        skipped.append(city)
        continue

    # Skip if already enriched and columns already present
    if all(c in tile_df.columns for c in NEW_COLS):
        print(f'[SKIP] {city}: new columns already present in sentinel')
        all_enriched.append(tile_df.assign(city=city))
        continue

    # Get ref file spec from tracker
    city_rows = tracker[tracker[_folder_col] == city]
    ref_spec = None
    if not city_rows.empty and _ref_col:
        ref_specs = city_rows[_ref_col].dropna().tolist()
        if ref_specs:
            ref_spec = '|'.join(
                p.strip() for raw in ref_specs
                for p in str(raw).split('|')
                if p.strip() not in ('', 'nan', 'None')
            ) or None

    # Load tiles
    tiles = load_tiles(city, DATA_DIR)
    if tiles is None:
        print(f'[SKIP] {city}: no tiles GPKG found')
        skipped.append(city)
        continue

    # Load reference
    ref = load_reference(city, DATA_DIR, ref_spec)
    if ref is None:
        print(f'[SKIP] {city}: no reference footprints found')
        skipped.append(city)
        continue

    # Enrich
    try:
        enriched_df = enrich_tile_metrics(tile_df, tiles, ref)
        enriched_df.to_parquet(enriched_path, index=False)
        n_tiles = len(enriched_df)
        mean_density = enriched_df['ref_building_density_per_km2'].mean()
        print(f'[OK]  {city}: {n_tiles} tiles | mean density={mean_density:.1f} bldg/km\u00b2')
        all_enriched.append(enriched_df.assign(city_=city))  # city col may already exist
    except Exception as e:
        print(f'[ERR] {city}: {e}')
        skipped.append(city)

# Combined CSV
if all_enriched:
    combined = pd.concat(all_enriched, ignore_index=True)
    csv_path = OUT_SCRATCH / 'per_tile_enriched_all_cities.csv'
    combined.to_csv(csv_path, index=False)
    print(f'\nCombined CSV \u2192 {csv_path}  ({len(combined)} rows, {combined["city"].nunique()} cities)')
    print(f'Skipped: {skipped}')

In [ ]:
if 'combined' not in dir() or combined is None:
    print('Run the enrichment cell first.')
else:
    # (1) Full column list
    print('=== Columns in enriched parquet ===')
    print(combined.columns.tolist())

    # (2) Summary stats per city for the 4 new columns
    print('\n=== Per-city summary stats ===')
    print(combined.groupby('city')[NEW_COLS].agg(['mean', 'std']).round(3).to_string())

    # (3) Scatter: ref_building_density_per_km2 vs f1, coloured by city
    fig, ax = plt.subplots(figsize=(10, 6))
    cities = combined['city'].unique()
    colours = cm.tab20(np.linspace(0, 1, len(cities)))
    for colour, city in zip(colours, sorted(cities)):
        grp = combined[combined['city'] == city]
        ax.scatter(
            grp['ref_building_density_per_km2'], grp['f1'],
            s=8, alpha=0.35, color=colour, label=city,
        )
    ax.set_xlabel('Reference building density (buildings / km\u00b2)')
    ax.set_ylabel('F1 score')
    ax.set_title('Reference building density vs F1 \u2014 all processed cities')
    ax.legend(markerscale=4, fontsize=7, ncol=2, loc='lower right')
    plt.tight_layout()
    scatter_path = OUT_SCRATCH / 'density_vs_f1_all_cities.png'
    plt.savefig(scatter_path, dpi=150)
    plt.show()
    print(f'Scatter saved \u2192 {scatter_path}')

## Notes

- **Non-destructive**: original `vector_metrics_tiles_all_datasets.parquet` is never modified.
- **Centroid-based assignment**: each reference building is assigned to the tile containing
  its centroid. Buildings straddling a tile boundary are counted only in the tile where the
  centroid falls, avoiding double-counting.
- **Future runs**: new cities processed after the pipeline update (commit `0daa7fb`) will
  already have these columns in their sentinel parquet and can be loaded directly.
- **Density units**: buildings per km² using the projected tile area (metres CRS), not
  geographic area — consistent with the pipeline's per-city density summary.